# Map wishlist file
When collecting several images to capture an AOI in Airbus OneAtlas, you can generate a "wishlist" file that contains the metadata for all the images you want to download. This file can be used to keep track of the images and their properties, and can also be used to automate the download process. The wishlist file is typically in KML format and includes information such as the image ID, acquisition date, cloud cover percentage, and other relevant metadata.

This notebook will read in the KML files and map them, focusing on the date and coverage properties


In [ ]:
import sys, os

import pandas as pd
import geopandas as gpd

import GOSTrocks.mapMisc as mapMisc

In [ ]:
wishlist_kml = r"c:\Users\WB411133\OneDrive - WBG\AAA_BPS\GOST\Projects\IMAGERY_RFP\Purchases\SPARC_EdAnderson\2026-04-07\KIGALI__wishlist.kml"

kml_layers = list(gpd.list_layers(wishlist_kml)['name'].values)
polygon_layer = gpd.read_file(wishlist_kml, layer=kml_layers[0], driver='KML')

all_layers = []
for layer_name in kml_layers[1:]:
    print(f"Layer: {layer_name}")
    inD = gpd.read_file(wishlist_kml, layer=layer_name, driver='KML')
    # extract the acquisition date and cloud cover from the description field
    total_description = inD['description'].iloc[0]
    for item in total_description.split("</td>"):
        if "acquisitionDate" in item:
            acquisition_date = item.split("<td>")[-1][:10]
        if "cloudCover" in item:
            cloud_cover = item.split("<td>")[-1]
    res = [acquisition_date, cloud_cover, inD['geometry'].iloc[0]]
    all_layers.append(pd.DataFrame([res], columns=['acquisition_date', 'cloud_cover', 'geometry']))

img_extents = gpd.GeoDataFrame(pd.concat(all_layers), geometry='geometry', crs=4326).reset_index()
img_extents = img_extents.sort_values('acquisition_date')


In [ ]:
img_extents['year'] = img_extents['acquisition_date'].str[:4]
img_extents

In [ ]:
img_extents